In [2]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

from scipy.stats import skew

np.random.seed(42)

In [3]:
# -------------------------------------------
# Homework 9 
# -------------------------------------------

# Define the supplied function

def simulate(A=1, B=1, C=10, D=1000):
    W = np.random.normal(0, 1, D)
    X = W + np.random.normal(0, B, D)
    Y = A*X - W + np.random.normal(0, C, D)
    return Y, X, W

In [ ]:
# Create a helper function that runs the regression

def run_regression(A=1, B=1, C=10, D=1000):
    Y, X, W = simulate(A=A, B=B, C=C, D=D)

    predictors = pd.DataFrame({
        "X": X,
        "W": W
    })

    predictors = sm.add_constant(predictors)

    model = sm.OLS(Y, predictors).fit()

    return model.params["X"], model.tvalues["X"]

# Result is considered statistically significant when: abs(t_value) > 1.96

In [ ]:
# Question 1 
# Run the simulation many times and calculate how often X is significant  

n_simulations = 10000

coefficients = []
t_values = []

for i in range(n_simulations):
    coefficient, t_value = run_regression(
        A=1,
        B=1,
        C=10,
        D=1000
    )

    coefficients.append(coefficient)
    t_values.append(t_value)

coefficients = np.array(coefficients)
t_values = np.array(t_values)

detection_probability = np.mean(np.abs(t_values) > 1.96)

print("Detection probability:", detection_probability)
print("Detection percentage:", detection_probability * 100)


Detection probability: 0.8853
Detection percentage: 88.53


Although the true coefficient is 1, random noise sometimes prevents the regression from finding statistical significance. With 1,000 observations, the effect is detected about 88% of the time.

In [6]:
# Question 2
# Calculate the skewness of the coefficient estimates 

coefficient_skew = skew(coefficients)

print("Skewness:", coefficient_skew)

Skewness: -0.015124264439696096


A skewness near zero means the estimated coefficients are distributed approximately symmetrically around the true coefficient of 1.

In [7]:
# Question 3 
# Vary B while keeping other parameters fixed 

B_values = [0.2, 0.6, 1.8, 5.4]

for B_value in B_values:
    detections = []

    for i in range(n_simulations):
        coefficient, t_value = run_regression(
            A=1,
            B=B_value,
            C=10,
            D=1000
        )

        detections.append(abs(t_value) > 1.96)

    probability = np.mean(detections)

    print(
        "B =", B_value,
        "| Detection probability =", round(probability, 3)
    )

B = 0.2 | Detection probability = 0.092
B = 0.6 | Detection probability = 0.468
B = 1.8 | Detection probability = 1.0
B = 5.4 | Detection probability = 1.0


Increasing B increases power because when B is very small, X is extremely similar to W. It is difficult for the regression to separate their individual effects. As B increases, X gains more variation that is independent of W. This makes the effect of X easier to identify.

X = W + np.random.normal(0, B, D)

In [8]:
# Question 4 
# Vary A

A_values = [0.5, 1.0, 2.0, 4.0]

for A_value in A_values:
    detections = []

    for i in range(n_simulations):
        coefficient, t_value = run_regression(
            A=A_value,
            B=1,
            C=10,
            D=100
        )

        detections.append(abs(t_value) > 1.96)

    probability = np.mean(detections)

    print(
        "A =", A_value,
        "| Detection probability =", round(probability, 3)
    )

A = 0.5 | Detection probability = 0.081
A = 1.0 | Detection probability = 0.166
A = 2.0 | Detection probability = 0.503
A = 4.0 | Detection probability = 0.972


In [ ]:
# Reflection 1: Heteroskadicity

# Define the heteroskadicity data generation process (DGP)
# Generate many completely new datasets

def simulate_heteroskedastic(n=500):
    X = np.random.normal(0, 1, n)

    # Error SD grows as the absolute value of X increases
    error_sd = 1 + 2*np.abs(X)
    error = np.random.normal(0, error_sd, n)

    Y = 2*X + error

    return Y, X

In [ ]:
# Estimate the true coefficient variability
# This is the empirical or “true” sampling standard deviation under the DGP

np.random.seed(42)

n_simulations = 5000
simulated_coefficients = []

for i in range(n_simulations):
    Y, X = simulate_heteroskedastic(n=500)

    X_model = sm.add_constant(X)
    model = sm.OLS(Y, X_model).fit()

    simulated_coefficients.append(model.params[1])

simulation_sd = np.std(simulated_coefficients, ddof=1)

print("Full simulation SD:", simulation_sd)

Full simulation SD: 0.1940766331091213


In [11]:
# Compare conventional and robust standard errors

Y, X = simulate_heteroskedastic(n=500)
X_model = sm.add_constant(X)

ordinary_model = sm.OLS(Y, X_model).fit()
robust_model = ordinary_model.get_robustcov_results(cov_type="HC3")

print("Estimated coefficient:", ordinary_model.params[1])
print("Full simulation SD:", simulation_sd)
print("Conventional OLS SE:", ordinary_model.bse[1])
print("HC3 robust SE:", robust_model.bse[1])

Estimated coefficient: 1.7758851286929094
Full simulation SD: 0.1940766331091213
Conventional OLS SE: 0.1384562326389495
HC3 robust SE: 0.239809780669609


The conventional OLS standard error assumes constant error variance. Because that assumption is violated, it will generally underestimate or otherwise incorrectly estimate the coefficient’s variability.

The HC3 robust standard error allows the errors to have different variances, so it should be closer to the full-simulation standard deviation.

In [12]:
# Reflection 2: Highly correlated or non-independent errors

# Define the correlated error DGP

def simulate_correlated_errors(n=500, rho=0.95):
    X = np.zeros(n)
    error = np.zeros(n)

    x_shocks = np.random.normal(0, 1, n)
    error_shocks = np.random.normal(0, 1, n)

    for t in range(1, n):
        # Persistent predictor
        X[t] = 0.8*X[t-1] + x_shocks[t]

        # Highly correlated errors
        error[t] = rho*error[t-1] + error_shocks[t]

    Y = 2*X + error

    return Y, X

In [13]:
# Estimate variability using full simulation

np.random.seed(42)

n_simulations = 5000
correlated_coefficients = []

for i in range(n_simulations):
    Y, X = simulate_correlated_errors(n=500, rho=0.95)

    X_model = sm.add_constant(X)
    model = sm.OLS(Y, X_model).fit()

    correlated_coefficients.append(model.params[1])

correlated_simulation_sd = np.std(
    correlated_coefficients,
    ddof=1
)

print("Full simulation SD:", correlated_simulation_sd)

Full simulation SD: 0.21858056565934542


In [14]:
# Compare conventional OLS and HAC standard errors

Y, X = simulate_correlated_errors(n=500, rho=0.95)
X_model = sm.add_constant(X)

ordinary_model = sm.OLS(Y, X_model).fit()

hac_model = ordinary_model.get_robustcov_results(
    cov_type="HAC",
    maxlags=10
)

print("Estimated coefficient:", ordinary_model.params[1])
print("Full simulation SD:", correlated_simulation_sd)
print("Conventional OLS SE:", ordinary_model.bse[1])
print("HAC robust SE:", hac_model.bse[1])

Estimated coefficient: 1.5326237731517356
Full simulation SD: 0.21858056565934542
Conventional OLS SE: 0.07424509014812235
HAC robust SE: 0.13679951656884365


The conventional OLS standard error treats all observations as independent, which is incorrect here. The HAC standard error accounts for dependence across nearby observations.

In [15]:
# Ordinary bootstrap comparison 

def ordinary_bootstrap_se(Y, X, n_bootstrap=2000):
    n = len(Y)
    bootstrap_coefficients = []

    for i in range(n_bootstrap):
        indices = np.random.choice(
            np.arange(n),
            size=n,
            replace=True
        )

        Y_boot = Y[indices]
        X_boot = X[indices]

        X_boot_model = sm.add_constant(X_boot)
        boot_model = sm.OLS(Y_boot, X_boot_model).fit()

        bootstrap_coefficients.append(boot_model.params[1])

    return np.std(bootstrap_coefficients, ddof=1)

In [16]:
# Run it on one dataset 

bootstrap_se = ordinary_bootstrap_se(
    Y,
    X,
    n_bootstrap=2000
)

print("Full simulation SD:", correlated_simulation_sd)
print("Conventional OLS SE:", ordinary_model.bse[1])
print("HAC SE:", hac_model.bse[1])
print("Ordinary bootstrap SE:", bootstrap_se)

Full simulation SD: 0.21858056565934542
Conventional OLS SE: 0.07424509014812235
HAC SE: 0.13679951656884365
Ordinary bootstrap SE: 0.07060342016001378


Ordinary bootstraps can fail because the ordinary bootstrap resamples individual observations and rearranges their order. That destroys the correlation between neighboring observations. It acts as if all rows were independent, so its estimated standard error may be too small.

In [17]:
# Block Boostrap

def block_bootstrap_se(
    Y,
    X,
    block_size=25,
    n_bootstrap=2000
):
    n = len(Y)
    bootstrap_coefficients = []

    for i in range(n_bootstrap):
        sampled_indices = []

        while len(sampled_indices) < n:
            start = np.random.randint(
                0,
                n - block_size + 1
            )

            block = np.arange(
                start,
                start + block_size
            )

            sampled_indices.extend(block)

        sampled_indices = np.array(sampled_indices[:n])

        Y_boot = Y[sampled_indices]
        X_boot = X[sampled_indices]

        X_boot_model = sm.add_constant(X_boot)
        boot_model = sm.OLS(Y_boot, X_boot_model).fit()

        bootstrap_coefficients.append(boot_model.params[1])

    return np.std(bootstrap_coefficients, ddof=1)

In [18]:
# Run the comparison

block_bootstrap_result = block_bootstrap_se(
    Y,
    X,
    block_size=25,
    n_bootstrap=2000
)

print("Full simulation SD:", correlated_simulation_sd)
print("Conventional OLS SE:", ordinary_model.bse[1])
print("HAC SE:", hac_model.bse[1])
print("Ordinary bootstrap SE:", bootstrap_se)
print("Block bootstrap SE:", block_bootstrap_result)

Full simulation SD: 0.21858056565934542
Conventional OLS SE: 0.07424509014812235
HAC SE: 0.13679951656884365
Ordinary bootstrap SE: 0.07060342016001378
Block bootstrap SE: 0.14014480755008984
